# AutoEIT Scoring Tutorial

## End-to-end automated scoring of learner transcriptions

This notebook demonstrates:
1. Loading transcriptions
2. Scoring against EIT rubric
3. Validating against human baseline
4. Computing protocol totals (20 items @ 6 pts each = 120)
5. Visualizing results and agreement metrics

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Import AutoEIT scoring components
from src.scoring.rubric import EITRubricEngine
from src.scoring.pipeline import ScoringPipeline
from src.scoring.validator import ScoringValidator

# Set up plotting
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print(f"✓ AutoEIT scoring ready")

## 2. Initialize Scoring Components

In [ ]:
# Create scoring components
rubric = EITRubricEngine()
pipeline = ScoringPipeline(use_ensemble=True)
validator = ScoringValidator()

print("✓ EITRubricEngine initialized")
print("✓ ScoringPipeline initialized (with ensemble enabled)")
print("✓ ScoringValidator initialized")

# Test with a sample
test_result = rubric.score("yo fui al mercado", "yo fui al mercado ayer")
print(f"\nQuick test: 'yo fui al mercado' vs 'yo fui al mercado ayer'")
print(f"  Score: {test_result.score}/6")
print(f"  Category: {test_result.category.name}")
print(f"  Confidence: {test_result.confidence:.2f}")
print(f"  Reasoning: {test_result.reasoning}")

## 3. Load and Prepare Transcription Data

In [ ]:
# Create sample transcription data
# In production, this would come from your transcription pipeline

sample_data = {
    'hypothesis': [
        'yo fui al mercado',
        'mi hermana estudia',
        'fui mercado ayer',
        'yo fui al mercado ayer con mi hermana',
        'el nino corriendo',
        'mi familia es grande',
        'ir escuela todos dias',
        'yo como pan cada manana',
        'gatos durmiendo',
        '',  # Empty response
        'nosotros habemos',  # Grammar error
        'ella come manzana roja'
    ],
    'reference': [
        'yo fui al mercado ayer',
        'mi hermana estudia en la universidad',
        'yo fui al mercado ayer',
        'yo fui al mercado ayer con mi hermana',
        'el nino esta corriendo',
        'mi familia es muy grande',
        'voy a la escuela todos los dias',
        'yo como pan cada manana',
        'los gatos estan durmiendo',
        'yo estoy aqui',
        'nosotros hemos comido',
        'ella come una manzana roja'
    ]
}

df = pd.DataFrame(sample_data)

print(f"Loaded {len(df)} transcriptions")
print(f"\nFirst 3 examples:")
print(df.head(3).to_string())

## 4. Score All Transcriptions

In [ ]:
# Score each transcription
scores = []
errors_list = []
reasonings = []
confidences = []
categories = []

for idx, row in df.iterrows():
    result = rubric.score(row['hypothesis'], row['reference'])
    scores.append(result.score)
    errors_list.append(', '.join(result.errors) if result.errors else 'None')
    reasonings.append(result.reasoning)
    confidences.append(result.confidence)
    categories.append(result.category.name)

df['auto_score'] = scores
df['errors'] = errors_list
df['reasoning'] = reasonings
df['confidence'] = confidences
df['category'] = categories

print(f"✓ Scored {len(df)} transcriptions")
print(f"\nScore distribution:")
print(df['auto_score'].value_counts().sort_index())
print(f"\nSample results:")
print(df[['hypothesis', 'reference', 'auto_score', 'category', 'confidence']].head(5).to_string())

## 5. Score Distribution Visualization

In [ ]:
# Plot score distribution
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

ax.hist(df['auto_score'], bins=7, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Score (0-6)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Automated EIT Scores', fontsize=14, fontweight='bold')
ax.set_xticks(range(7))
ax.grid(axis='y', alpha=0.3)

# Add count labels on bars
for i in range(7):
    count = (df['auto_score'] == i).sum()
    if count > 0:
        ax.text(i, count + 0.1, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Score statistics:")
print(f"  Mean: {df['auto_score'].mean():.2f}")
print(f"  Median: {df['auto_score'].median():.1f}")
print(f"  Std Dev: {df['auto_score'].std():.2f}")
print(f"  Min: {df['auto_score'].min()}")
print(f"  Max: {df['auto_score'].max()}")

## 6. Analyze Error Patterns

In [ ]:
# Extract error patterns
error_types = {}
for errors_str in df['errors']:
    if errors_str != 'None':
        for error in errors_str.split(', '):
            if error.strip():
                # Extract error type (first word)
                error_type = error.split(':')[0].strip()
                error_types[error_type] = error_types.get(error_type, 0) + 1

print(f"Error Pattern Analysis:")
print(f"\nMost common errors:")
for error_type, count in sorted(error_types.items(), key=lambda x: x[1], reverse=True):
    print(f"  {error_type}: {count}")

# Items with errors vs no errors
has_errors = df['errors'] != 'None'
print(f"\nItems with detected errors: {has_errors.sum()} / {len(df)} ({has_errors.sum()/len(df)*100:.1f}%)")
print(f"Items with no errors: {(~has_errors).sum()} / {len(df)} ({(~has_errors).sum()/len(df)*100:.1f}%)")

## 7. Simulate Human Baseline Scores

In [ ]:
# For demonstration, create a baseline with ~90% agreement
np.random.seed(42)

human_scores = df['auto_score'].copy()

# Introduce ~10% disagreement (random noise)
for i in range(len(human_scores)):
    if np.random.random() < 0.10:  # 10% of time
        # Adjust by +/- 1 point
        adjustment = np.random.choice([-1, 1])
        human_scores.iloc[i] = max(0, min(6, human_scores.iloc[i] + adjustment))

df['human_score'] = human_scores

print(f"✓ Created human baseline scores")
print(f"\nComparison (first 10):")
print(df[['hypothesis', 'auto_score', 'human_score']].head(10).to_string())

## 8. Validate Against Human Baseline

In [ ]:
# Item-level validation
auto_scores = df['auto_score'].tolist()
human_scores_list = df['human_score'].tolist()

item_metrics = validator.validate_against_baseline(auto_scores, human_scores_list)

print("\n" + "="*60)
print("ITEM-LEVEL AGREEMENT VALIDATION")
print("="*60)

print(f"\nMetrics:")
for key, value in item_metrics.items():
    if key != 'problem_cases':
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")

print(f"\nGoal Achievement:")
exact_agree = item_metrics['exact_agreement']
print(f"  Exact Agreement: {exact_agree:.1%}")
if exact_agree >= 0.90:
    print(f"    ✓ MEETS 90% TARGET")
else:
    print(f"    ⚠ Below target (need {0.90 - exact_agree:.1%} more)")

print(f"\n  Kappa (ordinal agreement): {item_metrics['kappa']:.3f}")
if item_metrics['kappa'] >= 0.80:
    print(f"    ✓ SUBSTANTIAL AGREEMENT")
else:
    print(f"    ⚠ Moderate agreement")

if item_metrics['problem_cases']:
    print(f"\n  Problem cases (diff > 1 point):")
    for case in item_metrics['problem_cases']:
        print(f"    Auto: {case['auto']}, Human: {case['human']} (diff: {case['diff']})")

## 9. Protocol-Level Agreement (EIT Protocol Scoring)

In [ ]:
# For protocol scoring: assume 2 protocols of 6 items each (in reality, 20 items per protocol)
# This demo uses smaller numbers for the sample data
protocol_metrics = validator.protocol_level_agreement(
    auto_scores, 
    human_scores_list,
    items_per_protocol=6  # Adjusted for demo data
)

print("\n" + "="*60)
print("PROTOCOL-LEVEL AGREEMENT (EIT Total Scoring)")
print("="*60)

print(f"\nMetrics:")
print(f"  Number of protocols: {protocol_metrics['n_protocols']}")
print(f"  Items per protocol: 6 (demo) / 20 (production)")
print(f"  Max total per protocol: {6 * 6} points (demo) / 120 points (production)")

print(f"\n  Protocol totals (Auto vs Human):")
for i, (auto_tot, human_tot) in enumerate(zip(
    protocol_metrics['protocol_totals_auto'],
    protocol_metrics['protocol_totals_human']
)):
    diff = protocol_metrics['protocol_diffs'][i]
    status = '✓' if diff <= 10 else '⚠'
    print(f"    Protocol {i+1}: Auto={auto_tot}, Human={human_tot}, Diff={diff} {status}")

print(f"\n  Mean protocol difference: {protocol_metrics['mean_protocol_diff']:.2f} points")
within_10 = protocol_metrics['within_10_pts']
print(f"  Within 10-point margin: {within_10:.1%}")
if within_10 >= 0.95:
    print(f"    ✓ MEETS TARGET")
else:
    print(f"    ⚠ Below target")

## 10. Agreement Heatmap

In [ ]:
# Create confusion matrix between auto and human scores
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(human_scores_list, auto_scores, labels=range(7))

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=range(7), 
    yticklabels=range(7),
    cbar_kws={'label': 'Count'}
)
ax.set_xlabel('Automatic System Score', fontsize=12)
ax.set_ylabel('Human Rater Score', fontsize=12)
ax.set_title('Agreement Matrix: Automatic vs Human Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Confusion matrix (diagonal = agreements):")
print(cm)

## 11. Confidence Analysis

In [ ]:
# Analyze confidence patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution
axes[0].hist(df['confidence'], bins=20, edgecolor='black', alpha=0.7, color='green')
axes[0].set_xlabel('Confidence Score (0-1)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Confidence Scores', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Confidence vs accuracy
differences = np.abs(np.array(auto_scores) - np.array(human_scores_list))
colors = ['green' if d == 0 else 'orange' if d <= 1 else 'red' for d in differences]

axes[1].scatter(df['confidence'], differences, c=colors, alpha=0.6, s=100)
axes[1].set_xlabel('Confidence Score', fontsize=12)
axes[1].set_ylabel('Difference from Human Score', fontsize=12)
axes[1].set_title('Confidence vs Scoring Accuracy', fontsize=12, fontweight='bold')
axes[1].axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Perfect match')
axes[1].axhline(y=1, color='orange', linestyle='--', alpha=0.5, label='Within 1 point')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Confidence statistics:")
print(f"  Mean: {np.mean(df['confidence']):.3f}")
print(f"  Median: {np.median(df['confidence']):.3f}")
print(f"  Min: {np.min(df['confidence']):.3f}")
print(f"  Max: {np.max(df['confidence']):.3f}")

low_conf = (df['confidence'] < 0.85).sum()
high_conf = (df['confidence'] >= 0.85).sum()
print(f"\n  Low confidence (<0.85): {low_conf} ({low_conf/len(df)*100:.1f}%)")
print(f"  High confidence (≥0.85): {high_conf} ({high_conf/len(df)*100:.1f}%)")

## 12. Summary Report

In [ ]:
print("\n" + "="*60)
print("FINAL SUMMARY REPORT")
print("="*60)

print(f"\nDataset:")
print(f"  Total items scored: {len(df)}")
print(f"  Protocols evaluated: {protocol_metrics['n_protocols']}")
print(f"  Items per protocol: 6 (demo)")

print(f"\nAccuracy Metrics:")
print(f"  Item-level exact agreement: {item_metrics['exact_agreement']:.1%}")
print(f"  Within-1-point agreement: {item_metrics['within_1_point']:.1%}")
print(f"  Cohen's Kappa: {item_metrics['kappa']:.3f}")
print(f"  Mean absolute diff: {item_metrics['mean_abs_diff']:.3f}")

print(f"\nGOSC Requirements Status:")
if item_metrics['exact_agreement'] >= 0.90:
    print(f"  ✓ Item-level agreement ≥90%: ACHIEVED")
else:
    print(f"  ⚠ Item-level agreement ≥90%: {item_metrics['exact_agreement']:.1%} (need {0.90 - item_metrics['exact_agreement']:.1%} more)")

if item_metrics['kappa'] >= 0.80:
    print(f"  ✓ Cohen's Kappa ≥0.80: ACHIEVED")
else:
    print(f"  ⚠ Cohen's Kappa ≥0.80: {item_metrics['kappa']:.3f}")

if protocol_metrics['within_10_pts'] >= 0.95:
    print(f"  ✓ Protocol-level within 10pts: ACHIEVED")
else:
    print(f"  ⚠ Protocol-level within 10pts: {protocol_metrics['within_10_pts']:.1%}")

print(f"\nScore Distribution:")
for score in range(7):
    count = (df['auto_score'] == score).sum()
    pct = count / len(df) * 100
    print(f"  Score {score}: {count:2d} items ({pct:5.1f}%)")

print(f"\nNext Steps:")
print(f"  1. Review items with >1 point difference (error analysis)")
print(f"  2. Validate on production transcription data")
print(f"  3. Fine-tune rubric rules if needed")
print(f"  4. Deploy to production pipeline")

print(f"\n" + "="*60)

## 13. Export Results

In [ ]:
# Export scored results
output_csv = 'scoring_results.csv'
df.to_csv(output_csv, index=False)
print(f"✓ Results exported to {output_csv}")

# Export validation metrics
validation_report = {
    'item_level': item_metrics,
    'protocol_level': {
        'n_protocols': protocol_metrics['n_protocols'],
        'mean_protocol_diff': protocol_metrics['mean_protocol_diff'],
        'within_10_pts': protocol_metrics['within_10_pts']
    },
    'summary': {
        'meets_90_percent_target': item_metrics['exact_agreement'] >= 0.90,
        'kappa_substantial': item_metrics['kappa'] >= 0.80
    }
}

output_json = 'validation_report.json'
with open(output_json, 'w') as f:
    json.dump(validation_report, f, indent=2)
print(f"✓ Validation report exported to {output_json}")

print(f"\nFiles created:")
print(f"  - {output_csv}")
print(f"  - {output_json}")